<a href="https://colab.research.google.com/github/janpfrang-hash/improved-data-evaluation-fatigue-tester/blob/main/Improved_Data_evaluation_fatigue_tester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
"""
S-N Curve Data Analysis for Google Colab - Multi-File Version
PRINT-OPTIMIZED VERSION for fast printing from Acrobat
Version - ENGLISH TRANSLATION WITH RASTERIZATION, DRIFT ANALYSIS & DIN 50100 CHECK
- Plots as raster graphics (not vectors)
- No transparencies (faster rendering)
- Optimized for Windows PDF printers
- Drift analysis with linear regression
- NEW: DIN 50100 conformity check (3% criterion for 3-sigma and 10-cycle avg)
- UPDATE: Now also supports the new CSV format (with Timestamp & Raw_Data)
- FIXED: Table column overlap (Filename truncation completely removed, column widths heavily expanded)
- FIXED: 'Distribution' and 'Resolution' columns removed from PDF exports
- ADDED: Overview table on the first page of the PDF (Font size reduced by 50%)
- FIXED: Cycle calculations updated for 10x logging interval
- REMOVED: Machine Parameters UI and Rasterization UI Text
- FIXED: Create Plots button renders inline again
- Version: 18 NOV 2026
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.gridspec import GridSpec
from io import StringIO
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
from datetime import datetime
from scipy import stats
from dataclasses import dataclass, field
from typing import Tuple, Optional, Dict

# ============================================================================
# PRINT OPTIMIZATION SETTINGS
# ============================================================================
SCREEN_DPI = 72
RASTER_DPI = 150
RASTERIZE_PLOTS = True
USE_TRANSPARENCY = False
MAX_PLOT_POINTS = 5000

# ============================================================================
# Global Variables
# ============================================================================
data_dict = {}
file_checkboxes = {}
cnf_dict = {}

COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
          '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']


# ============================================================================
# CNF Configuration Dataclass
# ============================================================================
@dataclass
class CNFConfig:
    kraft2_N: float         = 0.0
    kraftsoll2_N: float     = 0.0
    kraft1_N: float         = 0.0
    kraft1_ratio: int       = 0
    kraftabschaltung_N: float = 0.0
    toleranz_pct: int         = 0
    nrw_k1_mm: float        = 0.0
    nrw_k2_mm: float        = 0.0
    betaetigungen: int      = 0
    messen_alle_n: int      = 0
    device_version: str     = ""
    device_id: str          = ""
    display_id: str         = ""
    config_id: int          = 0
    raw_cnf: str            = ""
    parse_ok: bool          = False

    def force_tolerance_window(self) -> Tuple[float, float]:
        delta = self.kraft2_N * self.toleranz_pct / 100.0
        return (self.kraft2_N - delta, self.kraft2_N + delta)

def parse_cnf_line(raw: str) -> CNFConfig:
    cfg = CNFConfig(raw_cnf=raw)
    try:
        values = [int(x) for x in raw.replace('!', '').split(';')[1:]
                  if x.strip().lstrip('-').isdigit()]

        if len(values) < 32:
            return cfg

        def u16(i: int) -> int:
            return values[i] + (values[i + 1] << 8)

        def u32(i: int) -> int:
            return (values[i] + (values[i + 1] << 8) + (values[i + 2] << 16) + (values[i + 3] << 24))

        cfg.config_id           = values[0]
        cfg.kraft2_N            = u16(7)  / 10.0
        cfg.kraftsoll2_N        = u16(9)  / 10.0
        cfg.nrw_k1_mm           = u16(11) / 100.0
        cfg.nrw_k2_mm           = u16(13) / 100.0
        cfg.kraftabschaltung_N  = u16(15) / 10.0
        cfg.toleranz_pct        = values[17]
        cfg.messen_alle_n       = u16(21)
        cfg.kraft1_ratio        = values[23]
        cfg.betaetigungen       = u32(28)

        if cfg.kraft1_ratio > 0:
            cfg.kraft1_N = cfg.kraft2_N / cfg.kraft1_ratio
        cfg.parse_ok = True

    except Exception as e:
        print(f"CNFConfig parse error: {e}  -  raw: {raw[:80]}")
    return cfg

def parse_header_block(lines: list) -> CNFConfig:
    cfg = CNFConfig()
    for line in lines:
        line = line.strip()
        if '### VoiceCoilTestStand' in line:
            for token in line.split():
                if token.startswith('V') and len(token) > 1:
                    cfg.device_version = token
                if token.startswith('ID='):
                    cfg.device_id = token.split('=', 1)[1].rstrip('#').strip()
        elif line.startswith('For display'):
            parts = line.split()
            if len(parts) >= 3:
                cfg.display_id = parts[2]
        elif line.startswith('CNF;'):
            parsed = parse_cnf_line(line)
            if parsed.parse_ok:
                parsed.device_version = cfg.device_version
                parsed.device_id      = cfg.device_id
                parsed.display_id     = cfg.display_id
                cfg = parsed
    return cfg


# ============================================================================
# Downsampling
# ============================================================================
def downsample_data(df, max_points=MAX_PLOT_POINTS):
    if len(df) <= max_points:
        return df
    factor = len(df) // max_points + 1
    df_downsampled = df.iloc[::factor].copy()
    if df.index[0] not in df_downsampled.index:
        df_downsampled = pd.concat([df.iloc[[0]], df_downsampled])
    if df.index[-1] not in df_downsampled.index:
        df_downsampled = pd.concat([df_downsampled, df.iloc[[-1]]])
    return df_downsampled.sort_index()


# ============================================================================
# Parse Log File
# ============================================================================
def parse_logfile(file_content):
    lines = file_content.strip().split('\n')
    header_lines = []
    for line in lines:
        if line.startswith('DTA;') or line.startswith('Timestamp,'):
            break
        header_lines.append(line)

    cnf_config = parse_header_block(header_lines)
    dta_lines = [line for line in lines if 'DTA;' in line]
    data = []

    for line in dta_lines:
        dta_idx = line.find('DTA;')
        if dta_idx != -1:
            clean_line = line[dta_idx:]
            parts = clean_line.replace('DTA;', '').replace('!', '').split(';')
            if len(parts) >= 9:
                try:
                    row = {
                        'Cycles': int(parts[0]),
                        'Position_1_mm': int(parts[1]) / 100.0,
                        'Force_1_N': int(parts[2]) / 10.0,
                        'Additional_Travel_1_mm': int(parts[3]) / 100.0,
                        'Position_2_mm': int(parts[4]) / 100.0,
                        'Force_2_N': int(parts[5]) / 10.0,
                        'Additional_Travel_2_mm': int(parts[6]) / 100.0,
                        'Travel_mm': int(parts[7]) / 100.0,
                        'Error_Code': int(parts[8])
                    }
                    data.append(row)
                except (ValueError, IndexError):
                    continue

    df = pd.DataFrame(data)

    if len(df) > 0:
        df['R_Value'] = np.where(df['Force_2_N'] != 0, df['Force_1_N'] / df['Force_2_N'], np.nan)
        df['Force_Amplitude_N'] = (df['Force_2_N'] - df['Force_1_N']) / 2
        df['Mean_Force_N'] = (df['Force_2_N'] + df['Force_1_N']) / 2
        df['Force_Stroke_N'] = df['Force_2_N'] - df['Force_1_N']

        s_min = df['Position_1_mm'] + df['Additional_Travel_1_mm']
        s_max = df['Position_2_mm'] + df['Additional_Travel_2_mm']
        df['Travel_Stroke_mm'] = s_max - s_min

        df['Stiffness_N_per_mm'] = np.where(df['Travel_Stroke_mm'] != 0, df['Force_Stroke_N'] / df['Travel_Stroke_mm'], np.nan)
        window_size = 100
        df['Stiffness_Smoothed'] = df['Stiffness_N_per_mm'].rolling(window=window_size, center=True, min_periods=1).mean()
        initial_cycles = min(50, len(df))
        k_initial = df['Stiffness_Smoothed'].head(initial_cycles).mean()
        df['Stiffness_Normalized'] = df['Stiffness_Smoothed'] / k_initial
        df['Loss_of_Stiffness_Percent'] = (1 - df['Stiffness_Normalized']) * 100
        df.attrs['k_initial'] = k_initial

    return df, cnf_config


# ============================================================================
# Smooth Data
# ============================================================================
def smooth_data(df, window_size=50):
    df_smooth = df.copy()
    columns_to_smooth = [
        'Position_1_mm', 'Force_1_N', 'Additional_Travel_1_mm',
        'Position_2_mm', 'Force_2_N', 'Additional_Travel_2_mm',
        'Travel_mm', 'R_Value', 'Force_Amplitude_N', 'Mean_Force_N',
        'Loss_of_Stiffness_Percent'
    ]
    for col in columns_to_smooth:
        if col in df_smooth.columns:
            df_smooth[col] = df_smooth[col].rolling(window=window_size, center=True, min_periods=1).mean()
    return df_smooth


# ============================================================================
# Statistics & DIN 50100 Checks
# ============================================================================
def calculate_statistics(df, column_name):
    data = df[column_name].dropna()
    if len(data) == 0:
        return None

    min_val = data.min()
    max_val = data.max()
    mean_val = data.mean()
    std_val = data.std()
    var_val = data.var()

    upper_3sigma = mean_val + 3 * std_val
    lower_3sigma = mean_val - 3 * std_val

    # Logger only logs every 10th cycle, so each logged data point represents 10 cycles
    true_n = len(data) * 10

    return {
        'min': min_val, 'max': max_val, 'mean': mean_val,
        'std': std_val, 'var': var_val,
        'upper_3sigma': upper_3sigma, 'lower_3sigma': lower_3sigma,
        'n': true_n
    }

def evaluate_din50100_criteria(df, cnf):
    if not cnf or not cnf.parse_ok or cnf.kraft2_N == 0:
        return None
    target_fmax = cnf.kraft2_N
    allowed_dev = target_fmax * 0.03
    fmax_std = df['Force_2_N'].std()
    sigma3_val = 3 * fmax_std
    crit1_pass = sigma3_val <= allowed_dev

    # 10-cycle interval check: Each row represents 10 cycles, so evaluate raw deviations
    max_dev = (df['Force_2_N'] - target_fmax).abs().max()
    crit2_pass = max_dev <= allowed_dev

    return {
        'target': target_fmax,
        'allowed_dev': allowed_dev,
        '3sigma': sigma3_val,
        'crit1_pass': crit1_pass,
        'max_roll_dev': max_dev,
        'crit2_pass': crit2_pass
    }


# ============================================================================
# Drift Analysis
# ============================================================================
def calculate_drift(df, column_name):
    data = df[[column_name, 'Cycles']].dropna()
    if len(data) < 10:
        return None

    cycles = data['Cycles'].values
    values = data[column_name].values
    n_avg = min(50, len(data) // 10)
    start_value = values[:n_avg].mean()
    end_value = values[-n_avg:].mean()
    start_cycle = cycles[:n_avg].mean()
    end_cycle = cycles[-n_avg:].mean()

    try:
        slope, intercept, r_value, p_value, std_err = stats.linregress(cycles, values)
        r_squared = r_value ** 2
    except:
        slope, intercept, r_squared, p_value, std_err = 0, start_value, 0, 1, 0

    total_drift = end_value - start_value
    total_cycles = end_cycle - start_cycle

    relative_drift_percent = (total_drift / abs(start_value)) * 100 if start_value != 0 else 0
    drift_per_1000 = (total_drift / total_cycles) * 1000 if total_cycles > 0 else 0
    relative_drift_per_1000 = (relative_drift_percent / total_cycles) * 1000 if total_cycles > 0 else 0

    trend = "Stable" if abs(slope) < 1e-6 else ("Rising ↗" if slope > 0 else "Falling ↘")

    if r_squared > 0.8 and p_value < 0.001: significance = "Strong"
    elif r_squared > 0.5 and p_value < 0.05: significance = "Medium"
    elif r_squared > 0.3: significance = "Weak"
    else: significance = "Not significant"

    return {
        'start_value': start_value, 'end_value': end_value, 'total_drift': total_drift,
        'relative_drift_percent': relative_drift_percent, 'drift_per_1000': drift_per_1000,
        'relative_drift_per_1000': relative_drift_per_1000, 'slope': slope, 'intercept': intercept,
        'r_squared': r_squared, 'p_value': p_value, 'trend': trend, 'significance': significance,
        'total_cycles': total_cycles
    }


# ============================================================================
# Tables for Plots
# ============================================================================
def create_statistics_table(ax, selected_files, column_name, data_dict, use_smoothing=False):
    stats_list = []
    for filename in selected_files:
        df = data_dict[filename]
        if use_smoothing:
            df = smooth_data(df, window_size=50)
        s = calculate_statistics(df, column_name)
        if s:
            stats_list.append((filename, s))

    if not stats_list: return

    table_data = []
    headers = ['File', 'Min', 'Max', 'Mean', 'Std.Dev.', 'Variance', '-3σ', '+3σ', 'n']

    col_widths = [0.44, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.07]

    for filename, s in stats_list:
        short_name = filename
        decimal_places = 3 if column_name == 'R_Value' or abs(s['max']) < 10 else 2

        row = [
            short_name, f"{s['min']:.{decimal_places}f}", f"{s['max']:.{decimal_places}f}",
            f"{s['mean']:.{decimal_places}f}", f"{s['std']:.{decimal_places}f}", f"{s['var']:.{decimal_places}f}",
            f"{s['lower_3sigma']:.{decimal_places}f}", f"{s['upper_3sigma']:.{decimal_places}f}", f"{s['n']}"
        ]
        table_data.append(row)

    table = ax.table(cellText=table_data, colLabels=headers, colWidths=col_widths, cellLoc='center', loc='bottom', bbox=[0, -0.55, 1, 0.3])
    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1, 1.2)
    for i in range(len(headers)):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')
    for i in range(1, len(table_data) + 1):
        for j in range(len(headers)):
            table[(i, j)].set_facecolor('#E7E6E6' if i % 2 == 0 else '#FFFFFF')
            if j == 0:
                table[(i, j)].set_text_props(ha='left')

    pos = ax.get_position()
    ax.set_position([pos.x0, pos.y0 + 0.15, pos.width, pos.height * 0.75])


# ============================================================================
# Summary (Screen)
# ============================================================================
def create_summary_table(selected_files, data_dict, show_kriterium_10, show_kriterium_20, show_kriterium_50, use_smoothing=False):
    if not selected_files: return
    fig_height = 2 + 0.4 * len(selected_files)
    fig = plt.figure(figsize=(16, fig_height))
    ax = fig.add_subplot(111)
    ax.axis('off')

    table_data = []
    headers = ['File', 'Force 1 Ø [N]', 'Force 1 σ [N]', 'Force 2 Ø [N]', 'Force 2 σ [N]', 'DIN 3σ Check', 'DIN 10-Cyc. Check', 'F_a Ø [N]', 'F_a σ [N]']
    if show_kriterium_10: headers.append('Cycles @ 10%')
    if show_kriterium_20: headers.append('Cycles @ 20%')
    if show_kriterium_50: headers.append('Cycles @ 50%')

    for filename in selected_files:
        df = data_dict[filename]
        if use_smoothing: df = smooth_data(df, window_size=50)
        short_name = filename

        din_3sigma_str, din_10cyc_str = "---", "---"
        if filename in cnf_dict and cnf_dict[filename].parse_ok:
            din_eval = evaluate_din50100_criteria(df, cnf_dict[filename])
            if din_eval:
                din_3sigma_str = "PASS" if din_eval['crit1_pass'] else "FAIL"
                din_10cyc_str = "PASS" if din_eval['crit2_pass'] else "FAIL"

        row = [short_name, f"{df['Force_1_N'].mean():.2f}", f"{df['Force_1_N'].std():.2f}",
               f"{df['Force_2_N'].mean():.2f}", f"{df['Force_2_N'].std():.2f}", din_3sigma_str, din_10cyc_str,
               f"{df['Force_Amplitude_N'].mean():.2f}", f"{df['Force_Amplitude_N'].std():.2f}"]

        for thresh, show in [(10, show_kriterium_10), (20, show_kriterium_20), (50, show_kriterium_50)]:
            if show:
                idx = df[df['Loss_of_Stiffness_Percent'] >= thresh].first_valid_index()
                row.append(f"{int(df.loc[idx, 'Cycles']):,}" if idx is not None else "---")

        table_data.append(row)

    col_widths = [0.3] + [0.7 / (len(headers) - 1)] * (len(headers) - 1)
    table = ax.table(cellText=table_data, colLabels=headers, colWidths=col_widths, cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
    table.auto_set_font_size(False)
    table.set_fontsize(7)
    table.scale(1, 1.2)

    for i in range(len(headers)):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')

    for i in range(1, len(table_data) + 1):
        for j in range(len(headers)):
            table[(i, j)].set_facecolor('#E7E6E6' if i % 2 == 0 else '#FFFFFF')
            if j == 0:
                table[(i, j)].set_text_props(ha='left')
            if table_data[i-1][j] == "PASS": table[(i, j)].set_text_props(color='green', weight='bold')
            elif table_data[i-1][j] == "FAIL": table[(i, j)].set_text_props(color='red', weight='bold')

    title = f'Summary{" (smoothed)" if use_smoothing else ""}'
    fig.suptitle(title, fontsize=13, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()

# ============================================================================
# Combined Plots (Inline UI Rendering)
# ============================================================================
def create_combined_plots(show_pos1, show_kraft1, show_zusatz1,
                         show_pos2, show_kraft2, show_zusatz2, show_weg,
                         show_r_wert, show_kraftamplitude, show_mittelkraft,
                         show_steifigkeitsverlust, show_kriterium_10,
                         show_kriterium_20, show_kriterium_50, use_smoothing=False):
    selected_files = get_selected_files()
    if not selected_files:
        print("❌ No files selected.")
        return

    active_plots = sum([show_pos1, show_kraft1, show_zusatz1, show_pos2, show_kraft2,
                        show_zusatz2, show_weg, show_r_wert, show_kraftamplitude,
                        show_mittelkraft, show_steifigkeitsverlust])

    if active_plots == 0:
        print("❌ Please select at least one graph.")
        return

    fig, axes = plt.subplots(active_plots, 1, figsize=(14, 10 * active_plots), dpi=SCREEN_DPI)
    if active_plots == 1:
        axes = [axes]

    plot_idx = 0
    alpha_val = 0.7 if USE_TRANSPARENCY else 1.0

    configs = [
        (show_pos1, 'Position_1_mm', 'Position 1 [mm]', 'Position 1'),
        (show_kraft1, 'Force_1_N', 'Force 1 [N]', 'Force 1 (F_min)'),
        (show_zusatz1, 'Additional_Travel_1_mm', 'Add. Travel 1 [mm]', 'Additional Travel 1'),
        (show_pos2, 'Position_2_mm', 'Position 2 [mm]', 'Position 2'),
        (show_kraft2, 'Force_2_N', 'Force 2 [N]', 'Force 2 (F_max)'),
        (show_zusatz2, 'Additional_Travel_2_mm', 'Add. Travel 2 [mm]', 'Additional Travel 2'),
        (show_weg, 'Travel_mm', 'Travel [mm]', 'Travel'),
        (show_r_wert, 'R_Value', 'R-Value', 'R-Value'),
        (show_kraftamplitude, 'Force_Amplitude_N', 'F_a [N]', 'Force Amplitude'),
        (show_mittelkraft, 'Mean_Force_N', 'F_m [N]', 'Mean Force'),
        (show_steifigkeitsverlust, 'Loss_of_Stiffness_Percent', 'Loss [%]', 'Loss of Stiffness')
    ]

    for show, col, ylabel, title in configs:
        if show:
            for i, filename in enumerate(selected_files):
                df = data_dict[filename]
                if use_smoothing:
                    df = smooth_data(df, window_size=50)
                df_plot = downsample_data(df, MAX_PLOT_POINTS)
                axes[plot_idx].plot(df_plot['Cycles'], df_plot[col],
                                    color=COLORS[i % len(COLORS)],
                                    label=filename, alpha=alpha_val,
                                    linewidth=0.8)

            axes[plot_idx].set_xlabel('Cycles', fontsize=11)
            axes[plot_idx].set_ylabel(ylabel, fontsize=11)
            axes[plot_idx].set_title(title, fontsize=13, fontweight='bold')
            axes[plot_idx].grid(True, alpha=0.3, linewidth=0.5)
            axes[plot_idx].legend(fontsize=8)
            create_statistics_table(axes[plot_idx], selected_files, col, data_dict, use_smoothing)
            plot_idx += 1

    plt.subplots_adjust(hspace=0.7)
    plt.show()

    print("\nSUMMARY")
    create_summary_table(selected_files, data_dict, show_kriterium_10,
                        show_kriterium_20, show_kriterium_50, use_smoothing)

# ============================================================================
# File Upload & Selection
# ============================================================================
def upload_and_parse():
    global data_dict, file_checkboxes, cnf_dict
    print("Please select one or more log files...")
    uploaded = files.upload()
    if uploaded:
        files_loaded = 0
        for filename in uploaded.keys():
            content = uploaded[filename].decode('utf-8')
            print(f"\nParse file: {filename}")
            df, cnf_config = parse_logfile(content)
            if df is not None and len(df) > 0:
                data_dict[filename] = df
                cnf_dict[filename] = cnf_config
                files_loaded += 1
                print(f"  ✓ {len(df)} Data points")
        if files_loaded > 0:
            create_file_selection()
            enable_plot_controls()

def create_file_selection():
    global file_checkboxes
    file_checkboxes = {}
    for filename in data_dict.keys():
        checkbox = widgets.Checkbox(value=True, description=filename, style={'description_width': 'initial'}, layout=widgets.Layout(width='100%'))
        file_checkboxes[filename] = checkbox
    update_file_selection_ui()

def update_file_selection_ui():
    global file_selection_container
    if len(file_checkboxes) > 0:
        file_selection_container.children = [widgets.HTML("<h4>📁 Loaded Files:</h4>"), widgets.VBox(list(file_checkboxes.values()))]

def get_selected_files():
    return [filename for filename, checkbox in file_checkboxes.items() if checkbox.value]

def get_statistics_text():
    selected_files = get_selected_files()
    if not selected_files: return "No files selected"
    stats_text = "S-N CURVE DATA ANALYSIS\n" + "="*70 + "\n"
    for filename in selected_files:
        df = data_dict[filename]
        stats_text += f"\nFILE: {filename}\nCycles: {df['Cycles'].max()}\nLoss: {df['Loss_of_Stiffness_Percent'].iloc[-1]:.2f}%\n"

        if filename in cnf_dict and cnf_dict[filename].parse_ok:
            cfg = cnf_dict[filename]
            din_eval = evaluate_din50100_criteria(df, cfg)
            if din_eval:
                stats_text += f"\nDIN 50100 Check (Allowed Deviation ±3% = ±{din_eval['allowed_dev']:.2f} N):\n"
                stats_text += f"  1. 3-Sigma ({din_eval['3sigma']:.2f} N) <= 3% Limit: {'✅ PASS' if din_eval['crit1_pass'] else '❌ FAIL'}\n"
                stats_text += f"  2. Max 10-Cycle Avg Dev ({din_eval['max_roll_dev']:.2f} N) <= 3% Limit: {'✅ PASS' if din_eval['crit2_pass'] else '❌ FAIL'}\n"
    return stats_text


# ============================================================================
# PDF Export (Including new Overview Page)
# ============================================================================
def create_pdf_overview_page(pdf, selected_files, data_dict):
    """Creates an overview table as the very first page of the PDF export."""
    fig = plt.figure(figsize=(11, 8.5), dpi=SCREEN_DPI)
    fig.text(0.5, 0.9, 'DATA OVERVIEW', fontsize=18, fontweight='bold', ha='center')

    headers = ['Sample Name', 'Avg F_min [N]', 'Avg F_max [N]', 'Avg R-Value', 'Total Cycles', 'Std Dev F_max [N]', 'Std Dev F_max [%]']
    table_data = []

    for filename in selected_files:
        df = data_dict[filename]
        full_name = filename
        avg_fmin, avg_fmax, avg_rval = df['Force_1_N'].mean(), df['Force_2_N'].mean(), df['R_Value'].mean()
        tot_cycles, std_fmax = df['Cycles'].max(), df['Force_2_N'].std()
        std_fmax_pct = (std_fmax / avg_fmax) * 100 if avg_fmax != 0 else 0

        table_data.append([full_name, f"{avg_fmin:.2f}", f"{avg_fmax:.2f}", f"{avg_rval:.3f}", f"{int(tot_cycles):,}", f"{std_fmax:.2f}", f"{std_fmax_pct:.2f}%"])

    ax = fig.add_subplot(111)
    ax.axis('off')

    col_widths = [0.46, 0.09, 0.09, 0.09, 0.09, 0.09, 0.09]
    table = ax.table(cellText=table_data, colLabels=headers, colWidths=col_widths, cellLoc='center', loc='center', bbox=[0, 0.2, 1, 0.6])
    table.auto_set_font_size(False)
    table.set_fontsize(4.5)
    table.scale(1, 1.5)

    for i in range(len(headers)):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')
    for i in range(1, len(table_data) + 1):
        for j in range(len(headers)):
            table[(i, j)].set_facecolor('#E7E6E6' if i % 2 == 0 else '#FFFFFF')
            if j == 0:
                table[(i, j)].set_text_props(ha='left')

    pdf.savefig(fig, bbox_inches='tight', dpi=SCREEN_DPI)
    plt.close(fig)


def save_plots_and_data():
    selected_files = get_selected_files()
    if not selected_files:
        print("❌ No files selected.")
        return

    print("📄 Creating PRINT-OPTIMIZED PDF...")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    pdf_filename = f"SN_Curve_Comparison_{timestamp}_print.pdf"

    show_pos1, show_kraft1, show_zusatz1 = checkbox_pos1.value, checkbox_kraft1.value, checkbox_zusatz1.value
    show_pos2, show_kraft2, show_zusatz2 = checkbox_pos2.value, checkbox_kraft2.value, checkbox_zusatz2.value
    show_weg, show_r_wert = checkbox_weg.value, checkbox_r_wert.value
    show_kraftamplitude, show_mittelkraft = checkbox_kraftamplitude.value, checkbox_mittelkraft.value
    show_steifigkeitsverlust = checkbox_steifigkeitsverlust.value
    use_smoothing = checkbox_smooth.value

    active_plots = sum([show_pos1, show_kraft1, show_zusatz1, show_pos2, show_kraft2, show_zusatz2, show_weg, show_r_wert, show_kraftamplitude, show_mittelkraft, show_steifigkeitsverlust])

    with PdfPages(pdf_filename) as pdf:
        create_pdf_overview_page(pdf, selected_files, data_dict)

        stats_text = get_statistics_text()
        lines = stats_text.split('\n')
        lines_per_page = 55
        stats_pages = (len(lines) + lines_per_page - 1) // lines_per_page

        for page_start in range(0, len(lines), lines_per_page):
            page_lines = lines[page_start:page_start + lines_per_page]
            fig = plt.figure(figsize=(8.5, 11), dpi=SCREEN_DPI)
            fig.text(0.1, 0.95, 'STATISTICAL ANALYSIS - PRINT OPTIMIZED', fontsize=15, fontweight='bold', va='top')
            fig.text(0.1, 0.88, '\n'.join(page_lines), fontsize=7, family='monospace', va='top')
            plt.axis('off')
            pdf.savefig(fig, bbox_inches='tight', dpi=SCREEN_DPI)
            plt.close(fig)

        if active_plots > 0:
            fig, axes = plt.subplots(active_plots, 1, figsize=(14, 10 * active_plots), dpi=SCREEN_DPI)
            if active_plots == 1: axes = [axes]

            plot_idx = 0
            configs = [
                (show_pos1, 'Position_1_mm', 'Position 1 [mm]', 'Position 1'),
                (show_kraft1, 'Force_1_N', 'Force 1 [N]', 'Force 1 (F_min)'),
                (show_zusatz1, 'Additional_Travel_1_mm', 'Add. Travel 1 [mm]', 'Additional Travel 1'),
                (show_pos2, 'Position_2_mm', 'Position 2 [mm]', 'Position 2'),
                (show_kraft2, 'Force_2_N', 'Force 2 [N]', 'Force 2 (F_max)'),
                (show_zusatz2, 'Additional_Travel_2_mm', 'Add. Travel 2 [mm]', 'Additional Travel 2'),
                (show_weg, 'Travel_mm', 'Travel [mm]', 'Travel'),
                (show_r_wert, 'R_Value', 'R-Value', 'R-Value'),
                (show_kraftamplitude, 'Force_Amplitude_N', 'F_a [N]', 'Force Amplitude'),
                (show_mittelkraft, 'Mean_Force_N', 'F_m [N]', 'Mean Force'),
            ]

            for show, col, ylabel, title_base in configs:
                if show:
                    title = f'{title_base} over Cycles - Comparison' if len(selected_files) > 1 else f'{title_base} over Cycles'
                    for i, filename in enumerate(selected_files):
                        df = smooth_data(data_dict[filename], window_size=50) if use_smoothing else data_dict[filename]
                        df_plot = downsample_data(df, MAX_PLOT_POINTS)
                        axes[plot_idx].plot(df_plot['Cycles'], df_plot[col], color=COLORS[i % len(COLORS)], label=filename, linewidth=0.8, rasterized=RASTERIZE_PLOTS)

                    axes[plot_idx].set_xlabel('Cycles', fontsize=11)
                    axes[plot_idx].set_ylabel(ylabel, fontsize=11)
                    axes[plot_idx].set_title(title, fontsize=13, fontweight='bold')
                    axes[plot_idx].grid(True, alpha=0.3, linewidth=0.5)
                    axes[plot_idx].legend(fontsize=8)
                    create_statistics_table(axes[plot_idx], selected_files, col, data_dict, use_smoothing)
                    plot_idx += 1

            if show_steifigkeitsverlust:
                title = 'Loss of Stiffness over Cycles - Comparison' if len(selected_files) > 1 else 'Loss of Stiffness over Cycles'
                for i, filename in enumerate(selected_files):
                    df = smooth_data(data_dict[filename], window_size=50) if use_smoothing else data_dict[filename]
                    df_plot = downsample_data(df, MAX_PLOT_POINTS)
                    axes[plot_idx].plot(df_plot['Cycles'], df_plot['Loss_of_Stiffness_Percent'], color=COLORS[i % len(COLORS)], label=filename, linewidth=1.2, rasterized=RASTERIZE_PLOTS)

                axes[plot_idx].set_xlabel('Cycles', fontsize=11)
                axes[plot_idx].set_ylabel('Loss of Stiffness [%]', fontsize=11)
                axes[plot_idx].set_title(title, fontsize=13, fontweight='bold')
                axes[plot_idx].grid(True, alpha=0.3, linewidth=0.5)
                axes[plot_idx].legend(loc='best', fontsize=8)
                create_statistics_table(axes[plot_idx], selected_files, 'Loss_of_Stiffness_Percent', data_dict, use_smoothing)

            plt.subplots_adjust(hspace=0.7)
            pdf.savefig(fig, bbox_inches='tight', dpi=RASTER_DPI)
            plt.close(fig)

        d = pdf.infodict()
        d['Title'] = 'S-N Curve Data Analysis - PRINT OPTIMIZED'
        d['Author'] = 'Voice Coil Test Stand Analyzer'

    print(f"✅ PRINT-OPTIMIZED PDF: {pdf_filename}")
    files.download(pdf_filename)

# ============================================================================
# UI Definition
# ============================================================================
upload_button = widgets.Button(description='📁 Select Files', button_style='success', icon='upload', layout=widgets.Layout(width='450px', height='80px'))
upload_output = widgets.Output()
file_selection_container = widgets.VBox([widgets.HTML("<p><i>No files loaded yet</i></p>")])

# Defaults
checkbox_pos1 = widgets.Checkbox(value=False, description='Position 1 [mm]', disabled=True)
checkbox_kraft1 = widgets.Checkbox(value=True, description='Force 1 (F_min) [N]', disabled=True)
checkbox_zusatz1 = widgets.Checkbox(value=False, description='Add. Travel 1 [mm]', disabled=True)
checkbox_pos2 = widgets.Checkbox(value=False, description='Position 2 [mm]', disabled=True)
checkbox_kraft2 = widgets.Checkbox(value=True, description='Force 2 (F_max) [N]', disabled=True)
checkbox_zusatz2 = widgets.Checkbox(value=False, description='Add. Travel 2 [mm]', disabled=True)
checkbox_weg = widgets.Checkbox(value=True, description='Travel [mm]', disabled=True)
checkbox_r_wert = widgets.Checkbox(value=True, description='R-Value (σ_min/σ_max)', disabled=True)
checkbox_kraftamplitude = widgets.Checkbox(value=False, description='Force Amplitude F_a', disabled=True)
checkbox_mittelkraft = widgets.Checkbox(value=False, description='Mean Force F_m', disabled=True)
checkbox_steifigkeitsverlust = widgets.Checkbox(value=True, description='Loss of Stiffness [%]', disabled=True)

# Other defaults turned off
checkbox_kriterium_10 = widgets.Checkbox(value=False, description='10% Loss of Stiffness', disabled=True)
checkbox_kriterium_20 = widgets.Checkbox(value=False, description='20% Loss of Stiffness', disabled=True)
checkbox_kriterium_50 = widgets.Checkbox(value=False, description='50% Loss of Stiffness', disabled=True)
checkbox_smooth = widgets.Checkbox(value=False, description='Smooth Data', disabled=True, style={'description_width': 'initial'})


plot_button = widgets.Button(description='📊 Create Plots', button_style='primary', icon='chart-line', layout=widgets.Layout(width='220px', height='40px'), disabled=True)
save_button = widgets.Button(description='🖨️ PDF (Print-Optimized)', button_style='success', icon='print', layout=widgets.Layout(width='220px', height='40px'), disabled=True)
plot_output = widgets.Output()

def on_upload_button_clicked(b):
    with upload_output: clear_output(); upload_and_parse()

def on_plot_button_clicked(b):
    with plot_output:
        clear_output(wait=True)
        create_combined_plots(checkbox_pos1.value, checkbox_kraft1.value, checkbox_zusatz1.value,
                            checkbox_pos2.value, checkbox_kraft2.value, checkbox_zusatz2.value,
                            checkbox_weg.value, checkbox_r_wert.value, checkbox_kraftamplitude.value,
                            checkbox_mittelkraft.value, checkbox_steifigkeitsverlust.value,
                            checkbox_kriterium_10.value, checkbox_kriterium_20.value,
                            checkbox_kriterium_50.value, checkbox_smooth.value)

def on_save_button_clicked(b):
    save_plots_and_data()

def enable_plot_controls():
    for cb in [checkbox_pos1, checkbox_kraft1, checkbox_zusatz1, checkbox_pos2, checkbox_kraft2, checkbox_zusatz2, checkbox_weg, checkbox_r_wert, checkbox_kraftamplitude, checkbox_mittelkraft, checkbox_steifigkeitsverlust, checkbox_kriterium_10, checkbox_kriterium_20, checkbox_kriterium_50, checkbox_smooth]:
        cb.disabled = False
    plot_button.disabled = False
    save_button.disabled = False

upload_button.on_click(on_upload_button_clicked)
plot_button.on_click(on_plot_button_clicked)
save_button.on_click(on_save_button_clicked)

def create_ui():
    header = widgets.HTML(f"""
    <h2>🔬 S-N Curve Data Analysis - PRINT-OPTIMIZED</h2>
    <p style='background-color:#e3f2fd; padding:10px; border-radius:5px;'>
    <b>🔧 NEW: DIN 50100 Conformity Check & Cycle Adjustment</b><br>
    - Automatically runs a ±3% conformity check for F_max<br>
    - 'n' values correctly reflect the 10x logging interval<br>
    - Overview table generated on the first page of the PDF export
    </p><hr>
    """)

    messdaten_box = widgets.HBox([widgets.VBox([checkbox_pos1, checkbox_kraft1, checkbox_zusatz1]), widgets.VBox([checkbox_pos2, checkbox_kraft2, checkbox_zusatz2]), widgets.VBox([checkbox_weg])])
    woehler_box = widgets.HBox([checkbox_r_wert, checkbox_kraftamplitude, checkbox_mittelkraft])
    steifigkeit_box = widgets.VBox([checkbox_steifigkeitsverlust, widgets.HBox([checkbox_kriterium_10, checkbox_kriterium_20, checkbox_kriterium_50])])
    button_row = widgets.HBox([plot_button, save_button])

    ui = widgets.VBox([
        header, widgets.HTML("<h3>1️⃣ Load Files</h3>"), upload_button, upload_output,
        widgets.HTML("<br><h3>2️⃣ Select Files</h3>"), file_selection_container,
        widgets.HTML("<br><h3>3️⃣ Select Graphs</h3>"), widgets.HTML("<h4>📊 Measured Data</h4>"), messdaten_box,
        widgets.HTML("<h4>📈 Fatigue Parameters</h4>"), woehler_box, widgets.HTML("<h4>🔧 Stiffness</h4>"), steifigkeit_box,
        widgets.HTML("<br><h3>4️⃣ Output</h3>"), button_row, plot_output
    ])
    display(ui)

create_ui()

In [ ]:
# @title
# ==============================================================================
# Cell 1: Install required dependencies
# ==============================================================================
!pip install pypdf scipy matplotlib numpy ipywidgets -q

In [ ]:
# @title
# ==============================================================================
# Cell 2: Working S-N Curve Generator GUI with Download
# ==============================================================================
import io
import re
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pypdf import PdfReader
import ipywidgets as widgets
from IPython.display import display, clear_output

LATEST_PLOT_PATH = None

# ------------------------------------------------------------------------------
# 1. PDF Parser Function (Accurate Extraction)
# ------------------------------------------------------------------------------
def parse_pdf_data(file_bytes):
    reader = PdfReader(io.BytesIO(file_bytes))
    full_text = ""
    for page in reader.pages:
        full_text += (page.extract_text() or "") + "\n"

    pattern = re.compile(
        r'([a-zA-Z0-9_][a-zA-Z0-9_.\-\s\\(\\)]+?)\s+'
        r'(\d{1,2}\.\d{2,3})\s+'                      # F_min
        r'(\d{1,2}\.\d{2,3})\s+'                      # F_max
        r'(\d{1}\.\d{3})\s+'                          # R-value
        r'([\d,]{4,})'                                # Cycles
    )

    records = []
    seen = set()

    for m in pattern.finditer(full_text):
        raw_name = m.group(1).replace('\n', ' ').strip()

        # Discard table matches that bleed past .csv into other columns
        if '.csv' in raw_name.lower():
            parts = re.split(r'\.csv', raw_name, flags=re.IGNORECASE)
            after_csv = parts[1].strip() if len(parts) > 1 else ''
            if after_csv:
                continue

        # Strip any preceding table header/section title (e.g. "DATA OVERVIEW", "Force 1 ... Cycles")
        clean_name = re.sub(
            r'^.*?(?:Cycles|OVERVIEW|Name|Table|Loss|Travel|Force\s*\d*.*?Cycles)\s*',
            '',
            raw_name,
            flags=re.IGNORECASE | re.DOTALL
        ).strip()

        # Strip .csv and all trailing copy indices like (1), (2) down to the base specimen name
        clean_name = re.sub(r'\.csv$', '', clean_name, flags=re.IGNORECASE).strip()
        clean_name = re.sub(r'(\s*\\(\d+\\))+\s*$', '', clean_name).strip()

        if len(clean_name) < 3 or 'Sample' in clean_name:
            continue

        f_min = float(m.group(2))
        f_max = float(m.group(3))
        cycles = int(m.group(5).replace(',', ''))

        # Only retain true force tests (F_max >= 20 N; discards travel/R-value tables)
        if f_max < 20.0 or cycles < 500:
            continue

        # Deduplicate to strictly keep the 11 unique specimens
        if clean_name in seen:
            continue
        seen.add(clean_name)

        if '70N' in raw_name or f_max >= 64:
            nom_f = 70.0
        elif '60N' in raw_name or (54 <= f_max < 64):
            nom_f = 60.0
        elif '50N' in raw_name or f_max < 54:
            nom_f = 50.0
        else:
            nom_f = round(f_max, -1)

        records.append({
            'name': clean_name,
            'f_min': f_min,
            'f_max': f_max,
            'nominal_f': nom_f,
            'cycles': cycles
        })

    return records

# ------------------------------------------------------------------------------
# 2. ISO 12107 Evaluation & Plotting
# ------------------------------------------------------------------------------
def evaluate_and_plot(records, filename="Fatigue_Test", use_nominal=True):
    global LATEST_PLOT_PATH
    n = len(records)
    if n < 3:
        print("❌ Error: Fewer than 3 valid data points extracted.")
        return

    F = np.array([r['nominal_f'] if use_nominal else r['f_max'] for r in records])
    N = np.array([r['cycles'] for r in records])
    X = np.log10(F)
    Y = np.log10(N)

    X_bar = np.mean(X)
    Y_bar = np.mean(Y)
    S_xx = np.sum((X - X_bar)**2)
    S_xy = np.sum((X - X_bar) * (Y - Y_bar))
    S_yy = np.sum((Y - Y_bar)**2)

    b1 = S_xy / S_xx         # b1 = -k
    b0 = Y_bar - b1 * X_bar   # b0 = log10(C)
    k = -b1
    C = 10**b0

    nu = n - 2
    s = np.sqrt(max(0.0, (S_yy - b1 * S_xy) / nu))
    R2 = (S_xy**2) / (S_xx * S_yy) if S_xx * S_yy > 0 else 0

    t_crit = stats.t.ppf(0.975, df=nu)
    se_b1 = s / np.sqrt(S_xx)
    k_ci = (k - t_crit * se_b1, k + t_crit * se_b1)

    # Tolerance factor
    z_p = stats.norm.ppf(0.90)
    k_tol = stats.nct.ppf(0.95, df=nu, nc=z_p * np.sqrt(n)) / np.sqrt(n)

    # Print summary table
    print(f"✅ Extracted {n} valid specimens from: {filename}")
    print("=" * 72)
    print(f"{'Sample Name':<38} | {'F_nom [N]':<10} | {'F_max [N]':<10} | {'Cycles [N]':<10}")
    print("-" * 72)
    for r in records:
        print(f"{r['name']:<38} | {r['nominal_f']:<10.1f} | {r['f_max']:<10.2f} | {r['cycles']:<10,}")
    print("=" * 72)
    print(f"Fitted S-N: N = {C:.3e} * F^-{k:.2f}  |  k = {k:.2f} (95% CI: [{k_ci[0]:.2f}, {k_ci[1]:.2f}])  |  R² = {R2:.2f}\n")

    # Generate Figure
    fig, ax = plt.subplots(figsize=(10, 5.5), dpi=200)
    ax.set_facecolor('#ffffff')
    fig.patch.set_facecolor('#ffffff')

    F_plot = np.linspace(min(F) - 5, max(F) + 10, 200)
    X_plot = np.log10(F_plot)
    Y_fit = b0 + b1 * X_plot
    N_fit = 10**Y_fit

    N_p10 = 10**(Y_fit - 1.282 * s)
    N_p90 = 10**(Y_fit + 1.282 * s)
    h = np.sqrt(1.0 + 1.0/n + (X_plot - X_bar)**2 / S_xx)
    N_tol = 10**(Y_fit - k_tol * s * h)

    # Lines
    ax.plot(N_fit, F_plot, color='#1f77b4', lw=2.2, label='50% Failure Prob.')
    ax.plot(N_p10, F_plot, color='#d62728', lw=1.5, ls='--', label='10% Failure Prob.')
    ax.plot(N_p90, F_plot, color='#2ca02c', lw=1.5, ls='--', label='90% Failure Prob.')
    ax.plot(N_tol, F_plot, color='#800000', lw=1.8, ls=':', label='ISO 12107 Tol. Limit (10% / 95% Conf.)')

    # Gaussian distributions
    horizons = np.unique(F)
    bell_h = 3.5
    for h_load in horizons:
        sub_Y = Y[F == h_load]
        h_mean = np.mean(sub_Y)
        h_s = np.std(sub_Y, ddof=1) if len(sub_Y) > 1 else s
        y_pts = np.linspace(h_mean - 3.2 * h_s, h_mean + 3.2 * h_s, 100)
        pdf = stats.norm.pdf(y_pts, h_mean, h_s)
        pdf_scaled = (pdf / np.max(pdf)) * bell_h
        ax.fill_betweenx(h_load + pdf_scaled, 10**y_pts, 10**y_pts, color='#ba68c8', alpha=0.35, zorder=2)
        ax.plot(10**y_pts, h_load + pdf_scaled, color='#7b1fa2', lw=1.2, zorder=2)

    ax.scatter(N, F, color='black', s=40, zorder=5, label='Data Points')

    ax.set_xscale('log')
    ax.set_xlim(min(N) * 0.5, max(N) * 2.0)
    ax.set_ylim(min(F) - 7, max(F) + 10)
    ax.set_xlabel('Cycles to Failure [N]', fontsize=11, fontweight='bold')
    ax.set_ylabel('Force [N]', fontsize=11, fontweight='bold')
    ax.grid(True, which='both', ls='-', color='#e0e0e0', lw=0.6)

    # Info card
    stats_box = (
        f"$N = {C:.3e} \\cdot F^{{-{k:.2f}}}$\n"
        f"$k = {k:.2f}$ (95% CI: [{k_ci[0]:.2f}, {k_ci[1]:.2f}])\n"
        f"$\\sigma[\\log N] = {s:.3f}$\n"
        f"$R^2 = {R2:.2f}$\n"
        f"$n = {n}$"
    )
    ax.text(0.03, 0.95, stats_box, transform=ax.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='square,pad=0.5', facecolor='white', edgecolor='#cccccc', lw=1.0),
            fontsize=9.5)

    ax.set_title(f'S-N Curve according to ISO 12107 — {filename}', fontsize=12, fontweight='bold', pad=12)
    ax.legend(loc='lower left', framealpha=0.9, fontsize=8.5)

    plt.tight_layout()

    safe_name = re.sub(r'[^a-zA-Z0-9_\-]', '_', filename)
    LATEST_PLOT_PATH = f"SN_Curve_{safe_name}.png"
    plt.savefig(LATEST_PLOT_PATH, dpi=200, bbox_inches='tight')
    plt.show()

    download_button.disabled = False
    download_button.description = 'Download Curve (PNG)'

# ------------------------------------------------------------------------------
# 3. GUI Layout & Safe Event Handling
# ------------------------------------------------------------------------------
file_upload = widgets.FileUpload(
    accept='.pdf',
    multiple=False,
    description='Select PDF',
    tooltip='Select a logger summary PDF file'
)

force_type_toggle = widgets.RadioButtons(
    options=[('Nominal Force Horizons (e.g. 50N, 60N, 70N)', True),
             ('Actual Measured Peak Force (F_max)', False)],
    value=True,
    description='Evaluate on:',
    layout=widgets.Layout(width='auto')
)

plot_button = widgets.Button(
    description='Create S-N curve',
    button_style='primary',
    tooltip='Click to parse the PDF and generate the S-N curve',
    icon='line-chart',
    layout=widgets.Layout(width='200px', height='40px')
)

download_button = widgets.Button(
    description='Download Curve',
    button_style='success',
    tooltip='Download the generated S-N curve as a high-res PNG',
    icon='download',
    disabled=True,
    layout=widgets.Layout(width='220px', height='40px')
)

output_area = widgets.Output()

def on_plot_clicked(b):
    with output_area:
        clear_output()
        download_button.disabled = True

        if not file_upload.value:
            print("⚠️ Please select/upload a PDF file first using 'Select PDF'.")
            return

        uploaded_item = file_upload.value
        filename = "Uploaded_File.pdf"
        file_bytes = b""

        # Direct iterator unpacking — no dictionary key indexing, no lists
        if isinstance(uploaded_item, (tuple, list)):
            if not uploaded_item:
                print("⚠️ No file uploaded.")
                return
            entry = next(iter(uploaded_item))
            filename = entry.get('name', 'Uploaded_File.pdf')
            file_bytes = entry.get('content', b'')
        elif isinstance(uploaded_item, dict):
            if not uploaded_item:
                print("⚠️ No file uploaded.")
                return
            filename, val = next(iter(uploaded_item.items()))
            file_bytes = val['content'] if isinstance(val, dict) and 'content' in val else val
        else:
            print("❌ Error reading uploaded file format.")
            return

        if isinstance(file_bytes, memoryview):
            file_bytes = file_bytes.tobytes()
        else:
            file_bytes = bytes(file_bytes)

        print(f"⏳ Processing '{filename}'...")
        records = parse_pdf_data(file_bytes)

        if not records:
            print(f"❌ Could not find valid test data in '{filename}'.")
            return

        evaluate_and_plot(records, filename=filename.replace('.pdf', ''), use_nominal=force_type_toggle.value)

def on_download_clicked(b):
    global LATEST_PLOT_PATH
    if LATEST_PLOT_PATH and os.path.exists(LATEST_PLOT_PATH):
        try:
            from google.colab import files
            files.download(LATEST_PLOT_PATH)
        except Exception:
            with output_area:
                print(f"💾 Saved to disk: {LATEST_PLOT_PATH}")

plot_button.on_click(on_plot_clicked)
download_button.on_click(on_download_clicked)

header = widgets.HTML("<h3>📊 ISO 12107 S-N Curve Generator</h3><p>Upload a logger summary PDF, evaluate, and download the resulting curve.</p>")
controls = widgets.VBox([
    header,
    widgets.HBox([file_upload, plot_button, download_button], layout=widgets.Layout(align_items='center', gap='15px')),
    force_type_toggle
], layout=widgets.Layout(border='1px solid #ddd', padding='15px', border_radius='8px', margin='10px 0'))

display(controls, output_area)